# Notebook 6 — S3DIS Structural Element Extractor

**Data-prep file · runs first (once, before Notebook 1)**

> **Pipeline execution order: `6 (once) -> 1 -> 2 -> 4 -> 7 -> 8`.**
> A one-off data-prep step: parse the raw S3DIS `Area_3/` folders, keep only the
> structural-element points (wall / column / beam / door / window), and write a single
> `data/area3_structural.ply` for Notebook 1 to rasterise. Run it once, point `params.yaml`
> at the PLY it writes, then run the pipeline from Notebook 1.

## Purpose
Turn the raw S3DIS `Area_3/` annotation folders into one structural-only point cloud that
Notebook 1 treats as its input cloud.

## Inputs
- `data/Area_3/<room>/Annotations/<class>_<n>.txt` — per-element point files (`X Y Z R G B`).
  Only the structural classes `{wall, column, beam, door, window}` are kept.

## Outputs
- `data/area3_structural.ply` — XYZ-only structural point cloud (Open3D PLY).

## Assumptions
- S3DIS **Aligned Version**: all rooms already share one coordinate frame, so concatenating
  rooms needs no per-room transform. Up-axis is Z, coordinates in metres.
- Pure data prep — this notebook does **not** call `scan2bim.load_config()`.

### Setup
No `params.yaml`, no `load_config()`. `scan2bim.project_root()` locates the repo so the
`data/` paths resolve whether you Run-All from `notebooks/` or the project root.

In [1]:
import os
import numpy as np
import open3d as o3d
import scan2bim

ROOT = scan2bim.project_root()
AREA_DIR = os.path.join(ROOT, 'data', 'Area_3')
OUT_PLY = os.path.join(ROOT, 'data', 'area3_structural.ply')
STRUCTURAL = {'wall', 'column', 'beam', 'door', 'window'}
print('Area dir :', AREA_DIR, '| exists:', os.path.isdir(AREA_DIR))
print('out PLY  :', OUT_PLY)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Area dir : c:\onestruction\data\Area_3 | exists: True
out PLY  : c:\onestruction\data\area3_structural.ply


### Step 1 — Collect room folders (skip the alignment-angle file and other non-dirs)

In [2]:
rooms = sorted(d for d in os.listdir(AREA_DIR)
               if os.path.isdir(os.path.join(AREA_DIR, d)))
print('room folders:', len(rooms))
print(rooms)

room folders: 23
['WC_1', 'WC_2', 'conferenceRoom_1', 'hallway_1', 'hallway_2', 'hallway_3', 'hallway_4', 'hallway_5', 'hallway_6', 'lounge_1', 'lounge_2', 'office_1', 'office_10', 'office_2', 'office_3', 'office_4', 'office_5', 'office_6', 'office_7', 'office_8', 'office_9', 'storage_1', 'storage_2']


### Step 2 — Read structural-element annotations per room
Each `Annotations/<class>_<n>.txt` whose `<class>` is structural is loaded with
`numpy.loadtxt`; only XYZ (first 3 columns) is kept. `atleast_2d` guards single-point files.

In [3]:
chunks = []
for name in rooms:
    ann_dir = os.path.join(AREA_DIR, name, 'Annotations')
    if not os.path.isdir(ann_dir):
        continue
    kept = 0
    for f in sorted(os.listdir(ann_dir)):
        if not f.endswith('.txt') or f.split('_')[0] not in STRUCTURAL:
            continue
        arr = np.loadtxt(os.path.join(ann_dir, f), dtype=np.float32)
        if arr.size == 0:
            continue
        arr = np.atleast_2d(arr)[:, :3]
        chunks.append(arr)
        kept += len(arr)
    print('%-18s %9d structural points' % (name, kept))

WC_1                  506140 structural points
WC_2                  559863 structural points
conferenceRoom_1      359011 structural points
hallway_1             359943 structural points
hallway_2             524845 structural points
hallway_3              93277 structural points
hallway_4             846741 structural points
hallway_5              52669 structural points
hallway_6              96614 structural points
lounge_1              361746 structural points
lounge_2              638482 structural points
office_1              216583 structural points
office_10             206941 structural points
office_2              222680 structural points
office_3              341551 structural points
office_4              141290 structural points
office_5              372847 structural points
office_6              238946 structural points
office_7              210548 structural points
office_8              324503 structural points
office_9              243392 structural points
storage_1    

### Step 3 — Concatenate + report total count and bounding box

In [4]:
pts = np.concatenate(chunks, axis=0).astype(np.float32)
mn, mx = pts.min(axis=0), pts.max(axis=0)
print('total structural points: %d' % len(pts))
print('bbox min : [%.3f %.3f %.3f]' % tuple(mn))
print('bbox max : [%.3f %.3f %.3f]' % tuple(mx))
print('extent   : [%.3f %.3f %.3f] m' % tuple(mx - mn))

total structural points: 7034900
bbox min : [-2.587 -16.427 -0.049]
bbox max : [27.745 9.388 3.021]
extent   : [30.332 25.815 3.070] m


### Step 4 — Save as an Open3D PLY

In [5]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pts.astype(np.float64))
o3d.io.write_point_cloud(OUT_PLY, pcd)
print('wrote', len(pcd.points), 'points')

wrote 7034900 points


### Step 5 — Next: point `params.yaml` at this cloud, then run Notebook 1

In [6]:
print('packaged ->', OUT_PLY)
print()
print('Update params.yaml so the pipeline rasterises this cloud:')
print('  input:')
print('    file_path: data/area3_structural.ply')
print('    units_per_meter: 1.0          # S3DIS aligned coords are already in metres')
print('Then Run-All Notebook 1 -> 2 -> 4, then Notebook 7 -> 8.')

packaged -> c:\onestruction\data\area3_structural.ply

Update params.yaml so the pipeline rasterises this cloud:
  input:
    file_path: data/area3_structural.ply
    units_per_meter: 1.0          # S3DIS aligned coords are already in metres
Then Run-All Notebook 1 -> 2 -> 4, then Notebook 7 -> 8.
